<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
LangSmith-Project sicher löschen
</b></font> </br></p>

---


Ein LangSmith-Projekt enthält Traces, die beim Entwickeln schnell zahlreich werden. Für die Bereinigung braucht es deshalb zwei Wege: eine gezielte Auswahl einzelner Tracing-Projekte und eine bewusst bestätigte Gesamtlöschung. Dieses Notebook zeigt beide Wege mit einer kleinen Gradio-Oberfläche.

**Praxis-Transfer:** Beim Entwickeln des Meeting- & Research-Briefing-Agenten bleiben Test-Traces getrennt von produktiven Auswertungen und können nach einem Experiment gezielt entfernt werden.

In [ ]:
#@title 🛠️ Umgebung einrichten { display-mode: "form" }

!uv pip install --system -q gradio git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith-Variablen vor allen LangChain-Imports setzen.
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "m88-langsmith-tracing-loeschen"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

# Dieses Verwaltungs-Notebook adressiert keinen Agenten.
os.environ.pop("LANGSMITH_AGENT_ID", None)
os.environ.pop("LANGSMITH_AGENT_ENVIRONMENT", None)

# Tags und Metadaten für eventuelle Referenzläufe.
run_cfg = {
    "tags": ["m88", "langsmith", "verwaltung"],
    "metadata": {"notebook": "m88-langsmith-tracing-loeschen", "version": "1.0"},
}

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    JUDGE,
    PLANNER,
    WORKER,
    WORKER_PREMIUM,
    CODING,
    EMBEDDINGS,
)


# 1 | Löschablauf
---



Der Client lädt ausschließlich Projekte ohne Dataset-Verknüpfung. Vor dem Löschen werden Name und ID angezeigt. Die ID wird intern verwendet, weil Namen nicht zwingend eindeutig sind.

In [ ]:
#@markdown <p><font size="4" color='green'>Löschablauf</font></p>
diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TD
    A[Tracing-Projekte laden] --> B{Auswahl}
    B -->|Einzelne Projekte| C[IDs prüfen]
    B -->|Alle Projekte| D[Gesamtbestätigung prüfen]
    C --> E[Projekte löschen]
    D --> E
    E --> F[Liste aktualisieren]

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;

    %% Zuweisung
    class A input;
    class B metric;
    class F output;
'''
mermaid(diagram, width=700)

# 2 | LangSmith-Funktionen
---



Die Funktionen trennen das Lesen der Projektliste von der Löschaktion. Diese Trennung macht die Logik auch außerhalb der Gradio-Oberfläche testbar. Die Bestätigung ist absichtlich ein exakter Textvergleich; ein versehentliches Klicken oder eine leere Eingabe löst keine Löschung aus.


In [ ]:
import os
import gradio as gr
from langsmith import Client

client = Client(
    api_url=os.environ["LANGSMITH_ENDPOINT"],
    api_key=os.environ["LANGSMITH_API_KEY"],
)

def get_tracing_projects() -> list:
    """Lädt Projekte, die nicht mit einem Dataset verknüpft sind."""
    projects = list(client.list_projects(reference_free=True))
    return sorted(projects, key=lambda project: project.name.lower())

def project_choices(projects: list) -> list[tuple[str, str]]:
    """Erzeugt sichtbare Labels und stabile Projekt-IDs für die Auswahl."""
    return [(project.name, str(project.id)) for project in projects]

def checkbox_update(projects: list) -> gr.CheckboxGroup:
    """Erzeugt ein Update für die sichtbare Projektauswahl."""
    return gr.CheckboxGroup(choices=project_choices(projects), value=[])

def empty_checkbox_update() -> gr.CheckboxGroup:
    """Liefert bei einem Aktualisierungsfehler eine leere Auswahl."""
    return gr.CheckboxGroup(choices=[], value=[])

def toggle_all_projects(delete_all: bool) -> gr.CheckboxGroup:
    """Setzt oder leert alle sichtbaren Projekt-Checkboxen."""
    projects = get_tracing_projects()
    values = [str(project.id) for project in projects] if delete_all else []
    return gr.CheckboxGroup(choices=project_choices(projects), value=values)

def resolve_project_ids(selected_values: list[str]) -> list[str]:
    """Akzeptiert sowohl Checkbox-Werte als auch sichtbare Labels."""
    projects = get_tracing_projects()
    by_id = {str(project.id): str(project.id) for project in projects}
    by_label = {project.name: str(project.id) for project in projects}
    return [by_id.get(value, by_label.get(value, value)) for value in (selected_values or [])]

def resolve_targets(selected_values: list[str], delete_all: bool, projects: list) -> list[tuple[str, str]]:
    """Ermittelt unmittelbar vor der Aktion die tatsächlich zu löschenden Projekte."""
    if delete_all:
        return [(str(project.id), project.name) for project in projects]
    selected_ids = set(resolve_project_ids(selected_values))
    return [(str(project.id), project.name) for project in projects if str(project.id) in selected_ids]

def prepare_delete(selected_values: list[str], delete_all: bool) -> tuple[str, gr.Button, gr.Button]:
    """Zeigt eine zweite Löschschaltfläche für die finale Bestätigung an."""
    try:
        projects = get_tracing_projects()
        targets = resolve_targets(selected_values, delete_all, projects)
    except Exception as exc:
        return f"Fehler beim Prüfen der Projekte: {exc}", gr.Button(visible=False), gr.Button(visible=False)
    if not targets:
        return "Keine Projekte ausgewählt.", gr.Button(visible=False), gr.Button(visible=False)
    scope = "alle Projekte" if delete_all else f"{len(targets)} ausgewählte Projekte"
    return f"Bitte bestätigen: {scope} werden endgültig gelöscht.", gr.Button(visible=True), gr.Button(visible=True)

def cancel_delete() -> tuple[str, gr.Button, gr.Button, bool]:
    """Bricht die vorbereitete Löschung ohne API-Aufruf ab."""
    return "Löschung abgebrochen.", gr.Button(visible=False), gr.Button(visible=False), False

def delete_projects(selected_values: list[str], delete_all: bool) -> tuple[str, gr.CheckboxGroup, gr.Button, gr.Button, bool]:
    """Führt die endgültige Löschung mit robuster Fehleranzeige aus."""
    try:
        projects = get_tracing_projects()
        targets = resolve_targets(selected_values, delete_all, projects)
    except Exception as exc:
        return f"Fehler beim Laden der Löschziele: {exc}", empty_checkbox_update(), gr.Button(visible=False), gr.Button(visible=False), False
    if not targets:
        return "Keine Projekte ausgewählt.", checkbox_update(projects), gr.Button(visible=False), gr.Button(visible=False), False

    deleted = []
    errors = []
    for project_id, project_name in targets:
        try:
            client.delete_project(project_id=project_id)
            deleted.append(project_name)
        except Exception as exc:
            errors.append(f"{project_name}: {exc}")

    message = f"Gelöscht: {len(deleted)} von {len(targets)} Projekt(en)."
    if deleted:
        message += "\n" + "\n".join(f"- {name}" for name in deleted)
    if errors:
        message += "\nFehler:\n- " + "\n- ".join(errors)
    try:
        updated_choices = checkbox_update(get_tracing_projects())
    except Exception as exc:
        updated_choices = empty_checkbox_update()
        message += f"\nAktualisierung der Projektliste fehlgeschlagen: {exc}"
    return message, updated_choices, gr.Button(visible=False), gr.Button(visible=False), False


# 3 | Gradio-Oberfläche
---



Die Auswahl wird als Mehrfachauswahl dargestellt. Nach jeder Löschung wird die Liste automatisch neu geladen.

In [ ]:
#@markdown <p><font size="4" color='green'>Code</font></p>
try:
    initial_projects = get_tracing_projects()
    initial_status = ""
except Exception as exc:
    initial_projects = []
    initial_status = f"Fehler beim Laden der Projekte: {exc}"

custom_css = """
#project-selector .wrap {
    display: flex !important;
    flex-direction: column !important;
    align-items: stretch !important;
}
#project-selector .wrap label {
    width: 100% !important;
}
#cancel-prepare, #confirm-delete, #cancel-delete {
    background: #f97316 !important;
    border-color: #ea580c !important;
    color: white !important;
}
"""

# Lokales Farbschema im Stil von Sky/Mist — kein Hub-Zugriff erforderlich.
skymist = gr.themes.Soft(
    primary_hue="sky",
    secondary_hue="blue",
    neutral_hue="slate",
)

with gr.Blocks(title="LangSmith-Projekte verwalten", theme=skymist, css=custom_css) as demo:
    gr.Markdown("## LangSmith-Projekte verwalten")
    gr.Markdown("---")
    gr.Markdown("Mit dieser App lassen sich freie LangSmith-Tracing-Projekte auswählen und nach einer zweiten Bestätigung gezielt oder vollständig löschen.")
    selected_projects = gr.CheckboxGroup(
        choices=project_choices(initial_projects),
        label="Projekte für Löschung",
        elem_id="project-selector",
        interactive=True,
    )
    delete_all_checkbox = gr.Checkbox(label="Alle Projekte löschen")
    with gr.Row():
        prepare_button = gr.Button("Löschung vorbereiten", variant="primary")
        cancel_prepare_button = gr.Button("Abbrechen", elem_id="cancel-prepare")
    with gr.Row():
        confirm_button = gr.Button("Ja, löschen", elem_id="confirm-delete", visible=False)
        cancel_button = gr.Button("Abbrechen", elem_id="cancel-delete", visible=False)
    status = gr.Markdown(initial_status)

    delete_all_checkbox.change(
        fn=toggle_all_projects,
        inputs=delete_all_checkbox,
        outputs=selected_projects,
    )
    prepare_button.click(
        fn=prepare_delete,
        inputs=[selected_projects, delete_all_checkbox],
        outputs=[status, confirm_button, cancel_button],
    )
    cancel_prepare_button.click(
        fn=cancel_delete,
        outputs=[status, confirm_button, cancel_button, delete_all_checkbox],
    )
    confirm_button.click(
        fn=delete_projects,
        inputs=[selected_projects, delete_all_checkbox],
        outputs=[status, selected_projects, confirm_button, cancel_button, delete_all_checkbox],
    )
    cancel_button.click(
        fn=cancel_delete,
        outputs=[status, confirm_button, cancel_button, delete_all_checkbox],
    )

In [ ]:
# Aufruf
demo.launch(inbrowser=True, inline=False, prevent_thread_lock=True, debug=True)